In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
import os
import sys
import numpy as np
from torchvision.transforms.v2 import RandAugment
import torch
from tqdm.notebook import tqdm
from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../models"))

from dataset import get_data_loaders
from backbone import BackBone
from multiheadmodel import MultiHeadModel
from utils import deterministic, train, accuracy

In [34]:
backbone= BackBone()
backbone.load_state_dict(torch.load("../models/weights/backbone.pth"))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/home/alumno1/miniconda3/envs/vision/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/alumno1/miniconda3/envs/vision/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Comentario de claude para acelerar el entrenamiento de las cabezas:

Si el backbone está congelado, podés pre-calcular los embeddings una sola vez y entrenar solo sobre ellos — mucho más rápido

#### Fine Tuning Naive Task-IL

In [20]:
model = MultiHeadModel(backbone)
model.to(device)
for param in model.backbone.parameters(): 
    param.requires_grad = False
dataloaders = get_data_loaders(batch_size=512)

/home/alumno1/miniconda3/envs/vision/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


In [21]:
deterministic()
criterion = nn.CrossEntropyLoss()
for i in range(5):
    model.add_head(i)
    model.to(device)
    train_data = dataloaders[i][0]
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    epochs = 20

    train(model, train_data, optimizer, criterion, None, epochs, task_number=i, save=False)
    
    eval_data = dataloaders[i][1]
    acc = accuracy(model, eval_data, task_number=i)
    print(f"Task {i} Accuracy: {acc:.4f}")

model.save(f"../models/weights/naive_Task-IL.pth")

Evaluating: 100%|██████████| 2/2 [00:00<00:00, 12.03batch/s]


Task 0 Accuracy: 0.9610


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 11.85batch/s]


Task 1 Accuracy: 0.6070


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 12.10batch/s]


Task 2 Accuracy: 0.5260


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 12.04batch/s]


Task 3 Accuracy: 0.6450


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 12.07batch/s]


Task 4 Accuracy: 0.7630


#### Fine Tuning Naive Class-IL

In [71]:
model = MultiHeadModel(backbone)
model.to(device)
for param in model.backbone.parameters(): 
    param.requires_grad = False
dataloaders = get_data_loaders(batch_size=512)

Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


In [72]:
deterministic()
criterion = nn.CrossEntropyLoss()
for i in range(5):
    print(f"Training task {i}")
    if i == 0:
        model.add_head(i)
    else:
        model.expand_head()
    model.to(device)
    train_data = dataloaders[i][0]
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    epochs = 20

    train(model, train_data, optimizer, criterion, None, epochs, task_number=0, save=False, global_labels=True)
    
    for j in range(i + 1):
        eval_data = dataloaders[j][1]
        acc = accuracy(model, eval_data, task_number=0, global_labels=True)
        print(f"Task {j} Accuracy: {acc:.4f}")

model.save(f"../models/weights/naive_Class-IL.pth")

Training task 0


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 10.78batch/s]


Task 0 Accuracy: 0.9610
Training task 1


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 11.28batch/s]


Task 0 Accuracy: 0.0000


...

Task 2 Accuracy: 0.0000


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 10.81batch/s]


Task 3 Accuracy: 0.0000


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 11.31batch/s]


Task 4 Accuracy: 0.7540


### EWC TIL

In [66]:
from losses.ewc import EWCCriterion

In [67]:
model = MultiHeadModel(backbone)
model.to(device)
for param in model.backbone.parameters(): 
    param.requires_grad = False
dataloaders = get_data_loaders(batch_size=512)

Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


In [68]:
deterministic()
criterion = EWCCriterion(nn.CrossEntropyLoss(), lambda_=1)
for i in range(5):
    model.add_head(i)
    model.to(device)
    train_data = dataloaders[i][0]
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    epochs = 20

    train(model, train_data, optimizer, criterion, None, epochs, task_number=i, save=False)
    criterion.consolidate(model, dataloaders[i][0], task_number=i)
    
    eval_data = dataloaders[i][1]
    acc = accuracy(model, eval_data, task_number=i)
    print(f"Task {i} Accuracy: {acc:.4f}")

model.save(f"../models/weights/naive_Task-IL.pth")

Evaluating: 100%|██████████| 2/2 [00:00<00:00, 10.32batch/s]


Task 0 Accuracy: 0.9610


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 11.23batch/s]


Task 1 Accuracy: 0.5890


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 10.85batch/s]


Task 2 Accuracy: 0.5370


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 11.10batch/s]


Task 3 Accuracy: 0.5530


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 10.76batch/s]


Task 4 Accuracy: 0.7700


### EWC CIL

In [79]:
model = MultiHeadModel(backbone)
model.to(device)
for param in model.backbone.parameters(): 
    param.requires_grad = False
dataloaders = get_data_loaders(batch_size=512)

Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


In [80]:
deterministic()
criterion = EWCCriterion(nn.CrossEntropyLoss(), lambda_=1, global_labels=True)
for i in range(5):
    print(f"Training task {i}")
    if i == 0:
        model.add_head(i)
    else:
        model.expand_head()
    model.to(device)
    train_data = dataloaders[i][0]
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    epochs = 20

    train(model, train_data, optimizer, criterion, None, epochs, task_number=0, save=False, global_labels=True)
    criterion.consolidate(model, dataloaders[i][0], task_number=0)

    if i == 1:
        model.eval()
        x, y = next(iter(dataloaders[0][1]))
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            pred = model(x, 0)
            print("Logits shape:", pred.shape)        # debería ser (batch, 4)
            print("Logits mean:", pred.mean(dim=0))   # ¿los primeros 2 están aplastados?
            print("Predicted classes:", pred.argmax(dim=1)[:10])  # ¿predice siempre 2 o 3?
    
    for j in range(i + 1):
        eval_data = dataloaders[j][1]
        acc = accuracy(model, eval_data, task_number=0, global_labels=True)
        print(f"Task {j} Accuracy: {acc:.4f}")

model.save(f"../models/weights/naive_Class-IL.pth")

Training task 0


Epochs: 100%|██████████| 20/20 [00:10<00:00,  1.94epoch/s, loss=0.00395] 


heads.0.weight: grad_norm=1.586387
heads.0.bias: grad_norm=0.002233
heads.0.weight: grad_norm=36.346237
heads.0.bias: grad_norm=0.005549
heads.0.weight: grad_norm=10.436065
heads.0.bias: grad_norm=0.002515
heads.0.weight: grad_norm=0.003050
heads.0.bias: grad_norm=0.000005
heads.0.weight: grad_norm=8.165030
heads.0.bias: grad_norm=0.006828
heads.0.weight: grad_norm=12.755463
heads.0.bias: grad_norm=0.002233
heads.0.weight: grad_norm=0.000003
heads.0.bias: grad_norm=0.000000
heads.0.weight: grad_norm=0.061909
heads.0.bias: grad_norm=0.000085
heads.0.weight: grad_norm=0.000000
heads.0.bias: grad_norm=0.000000
heads.0.weight: grad_norm=23.576241
heads.0.bias: grad_norm=0.000145
heads.0.weight: grad_norm=1.904760
heads.0.bias: grad_norm=0.002496
heads.0.weight: grad_norm=0.001719
heads.0.bias: grad_norm=0.000002
heads.0.weight: grad_norm=0.001391
heads.0.bias: grad_norm=0.000002
heads.0.weight: grad_norm=2.480018
heads.0.bias: grad_norm=0.002726
heads.0.weight: grad_norm=24.215307
heads.0.

Evaluating: 100%|██████████| 2/2 [00:00<00:00, 10.78batch/s]


Task 0 Accuracy: 0.9610
Training task 1


...

Task 2 Accuracy: 0.0000


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 10.99batch/s]


Task 3 Accuracy: 0.0000


Evaluating: 100%|██████████| 2/2 [00:00<00:00, 11.01batch/s]


Task 4 Accuracy: 0.7330


### LwF TIL